<a href="https://colab.research.google.com/github/balaa1407/ai-guardrail-waf/blob/main/train_safeguard_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ Train AI Guardrail (Gemma 2B) on T4 GPU
This notebook fine-tunes Google's Gemma-2B-it using 4-bit QLoRA to act as a Brand-Safety Checker.

In [1]:
!pip install -q -U transformers peft bitsandbytes trl datasets accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# Gemma requires you to accept their license on HuggingFace first.
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
# Load the dataset we generated
dataset = load_dataset('json', data_files='security_dataset.json', split='train')

def format_instruction(example):
    return f"Instruction:\n{example['instruction']}\n\nInput:\n{example['input']}\n\nOutput:\n{example['output']}"

# We don't need to map yet, SFTTrainer handles formatting

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
# 4-Bit Quantization Configuration (Squeezes model into T4 GPU)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_id = "google/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = 'right'
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [5]:
# LoRA Configuration (We only train ~1% of the model's brain to save time)
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gemma-guardrail-results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=20,
    logging_steps=5,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    max_grad_norm=0.3,
    max_steps=100,
    warmup_steps=3,
    lr_scheduler_type="constant"
)

# 1. Format the text ourselves to bypass the bug!
def format_data(example):
    example["text"] = f"Instruction:\n{example['instruction']}\n\nInput:\n{example['input']}\n\nOutput:\n{example['output']}"
    return example

formatted_dataset = dataset.map(format_data)

# 2. Tell the trainer to just use our perfect "text" column
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,

    args=training_args,
)

# Start Training!
trainer.train()



/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:964: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  args = SFTConfig(**dict_args)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass u

Step,Training Loss
5,2.579807
10,0.712361
15,0.309730
20,0.171320
25,0.076842
30,0.068278
35,0.054574
40,0.066518
45,0.047219
50,0.046031


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=100, training_loss=0.2279890449345112, metrics={'train_runtime': 330.8061, 'train_samples_per_second': 2.418, 'train_steps_per_second': 0.302, 'total_flos': 1401847192043520.0, 'train_loss': 0.2279890449345112, 'epoch': 5.266666666666667})

In [19]:
# Save the fine-tuned adapters
trainer.model.save_pretrained("gemma-guardrail-adapter")
tokenizer.save_pretrained("gemma-guardrail-adapter")
print("Model saved! You can now download the 'gemma-guardrail-adapter' folder.")

Model saved! You can now download the 'gemma-guardrail-adapter' folder.


In [20]:
!zip -r gemma-guardrail-adapter.zip gemma-guardrail-adapter

  adding: gemma-guardrail-adapter/ (stored 0%)
  adding: gemma-guardrail-adapter/tokenizer_config.json (deflated 50%)
  adding: gemma-guardrail-adapter/adapter_config.json (deflated 60%)
  adding: gemma-guardrail-adapter/README.md (deflated 65%)
  adding: gemma-guardrail-adapter/adapter_model.safetensors (deflated 21%)
  adding: gemma-guardrail-adapter/chat_template.jinja (deflated 52%)
  adding: gemma-guardrail-adapter/tokenizer.json (deflated 84%)
